# Brunel SNN

Faithful split of HOLYKALE's recurrent network. The full 125,000-neuron simulation is opt-in: set `RUN_SIMULATION = True` only for a deliberate reproduction run.

## Purpose, equations, and assumptions

We simulate leaky integrate-and-fire neurons with

$$\tau \frac{dv}{dt}=-(v-R_I),$$

Euler integration, threshold $v>V_T$, reset $V_R$, and a 2 ms absolute refractory period. Excitatory events add $J$ and inhibitory events add $-gJ$ after a common delay. Every target receives exactly $C_E$ excitatory and $C_I$ inhibitory incoming synapses. Sources are sampled with replacement, so duplicate edges and autapses are retained exactly as in HOLYKALE.

The saved original spectrum calculation is retained for traceability. The default analysis uses a memory-bounded batched FFT instead of constructing the full neuron-by-time matrix.

## Imports, parameters, and random seeds

In [1]:
from pathlib import Path
import json
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import __version__ as scipy_version
from scipy.fft import fft

try:
    from brian2 import (NeuronGroup, PopulationRateMonitor, SpikeMonitor,
                        Synapses, defaultclock, mV, ms, prefs, run, seed,
                        start_scope, second)
    from brian2 import __version__ as brian2_version
except ImportError as exc:
    raise ImportError("Install requirements.txt before running the SNN notebook.") from exc

# Editable controls. False prevents an accidental full-memory run.
RUN_SIMULATION = False
USE_CYTHON = True
SAVE_FIGURES = True
RUN_ID = "holy kale-faithful-snn"

# --- Population sizes -------------------------------------------------------
N_E      = 100000                    # excitatory neurons
N_I      = 25000                     # inhibitory neurons
N        = N_E + N_I                 # total = 125000 neurons

# --- Fixed incoming connectivity --------------------------------------------
C_E      = 1000                      # excitatory inputs per neuron
C_I      = 250                       # inhibitory inputs per neuron

# --- Neuron parameters ------------------------------------------------------
tau     = 20 * ms                    # membrane time constant
theta   = 20 * mV                    # firing threshold
V_r     = 10 * mV                    # reset potential
tau_rp  = 2 * ms                     # absolute refractory period

# --- Synapse and external-input parameters ----------------------------------
J       = 0.1 * mV                   # excitatory PSP amplitude
g       = 4.0                        # relative inhibitory strength
D       = 1.5 * ms                   # synaptic transmission delay
RI      = 30 * mV                    # constant external input

# --- Simulation and analysis parameters -------------------------------------
dt       = 0.1 * ms                  # Euler step
sim_time = 3000 * ms                 # measurement duration
SEED     = 4321
SPECTRUM_BATCH_SIZE = 256
SMALL_TEST_CONFIG = {"N_E": 20, "N_I": 5, "C_E": 4, "C_I": 2, "duration_ms": 20.0}
# SMALL_TEST_CONFIG is for explicit checks only; it does not replace the defaults.

# Units: N and C are neuron/synapse counts; tau, D, tau_rp, dt, and
# sim_time are times; voltages and J/RI are mV; g is dimensionless.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "01_Original_Colab").is_dir()), Path.cwd())
RESULTS = ROOT / "02_Brunel_SNN" / "results"
FIGURES = ROOT / "02_Brunel_SNN" / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
if USE_CYTHON:
    prefs.codegen.target = "cython"


## Neuron populations and recurrent connectivity

In [2]:
def build_network():
    """Build HOLYKALE's LIF network and fixed-in-degree replacement sampling."""
    start_scope()
    defaultclock.dt = dt
    seed(SEED)  # preserves Brian2/NumPy initialization and random streams
    neurons = NeuronGroup(
        N,
        model="dv/dt = (-v + RI) / tau : volt (unless refractory)",
        threshold="v > theta",
        reset="v = V_r",
        refractory=tau_rp,
        method="euler",
    )
    neurons.v = "rand() * mV"
    exc = neurons[:N_E]
    inh = neurons[N_E:]
    exc_syn = Synapses(exc, neurons, on_pre="v += J", delay=D)
    inh_syn = Synapses(inh, neurons, on_pre="v += -g*J", delay=D)
    targets_e = np.repeat(np.arange(N, dtype=np.int32), C_E)
    targets_i = np.repeat(np.arange(N, dtype=np.int32), C_I)
    # randint is intentional: fixed incoming counts, with replacement.
    sources_e = np.random.randint(0, N_E, size=N * C_E)
    sources_i = np.random.randint(0, N_I, size=N * C_I)
    exc_syn.connect(i=sources_e, j=targets_e)
    inh_syn.connect(i=sources_i, j=targets_i)
    return neurons


## Monitors and simulation

In [3]:
def run_network():
    neurons = build_network()
    rate_monitor = PopulationRateMonitor(neurons)
    spike_monitor = SpikeMonitor(neurons)
    started = time.perf_counter()
    run(sim_time)
    return neurons, rate_monitor, spike_monitor, time.perf_counter() - started

if RUN_SIMULATION:
    neurons, rate_monitor, spike_monitor, runtime_s = run_network()
else:
    print("Simulation disabled. Set RUN_SIMULATION = True for the full network.")


Simulation disabled. Set RUN_SIMULATION = True for the full network.


## Firing rates and ISI CV

In [4]:
def rate_and_cv(spike_indices, spike_times_s, n_neurons, duration_s):
    counts = np.bincount(spike_indices, minlength=n_neurons)
    rates = counts / duration_s
    cvs = []
    for neuron in range(n_neurons):
        times = spike_times_s[spike_indices == neuron]
        if times.size > 2:
            isi = np.diff(times)
            cvs.append(np.std(isi) / np.mean(isi))
    return float(np.mean(rates)), float(np.mean(cvs)) if cvs else float("nan"), rates, np.asarray(cvs)

if RUN_SIMULATION:
    spike_i = np.asarray(spike_monitor.i[:], dtype=np.int32)
    spike_t_s = np.asarray(spike_monitor.t[:] / second)
    mean_rate, mean_cv, neuron_rates, cv_values = rate_and_cv(
        spike_i, spike_t_s, N, float(sim_time / second))
    print(f"Mean firing rate: {mean_rate:.3f} Hz; mean ISI CV: {mean_cv:.3f}")


## Single-neuron power spectra

In [5]:
def spectrum_original_dense(spike_indices, spike_times_s, n_neurons, duration_s, bin_s):
    """HOLYKALE reference: dense matrix, FFT, and division by two."""
    n_bins = int(duration_s / bin_s)
    matrix = np.zeros((n_neurons, n_bins), dtype=np.float64)
    bins = np.asarray(spike_times_s / bin_s, dtype=np.int64)
    valid = (bins >= 0) & (bins < n_bins)
    matrix[spike_indices[valid], bins[valid]] = 1.0
    transform = np.fft.fft(matrix, axis=1)
    power = np.abs(transform) ** 2 / 2.0
    return np.arange(n_bins) / duration_s, np.mean(power, axis=0)

def spectrum_batched(spike_indices, spike_times_s, n_neurons, duration_s, bin_s, batch_size):
    """Equivalent accumulation without the N-by-time dense allocation."""
    n_bins = int(duration_s / bin_s)
    sums = np.zeros(n_bins, dtype=np.float64)
    order = np.argsort(spike_indices, kind="stable")
    indices = spike_indices[order]
    times = spike_times_s[order]
    for first in range(0, n_neurons, batch_size):
        last = min(first + batch_size, n_neurons)
        block = np.zeros((last - first, n_bins), dtype=np.float64)
        mask = (indices >= first) & (indices < last)
        bins = np.asarray(times[mask] / bin_s, dtype=np.int64)
        valid = (bins >= 0) & (bins < n_bins)
        block[indices[mask][valid] - first, bins[valid]] = 1.0
        sums += (np.abs(np.fft.fft(block, axis=1)) ** 2).sum(axis=0) / 2.0
        del block
    return np.arange(n_bins) / duration_s, sums / n_neurons

if RUN_SIMULATION:
    frequency_hz, spectrum = spectrum_batched(
        spike_i, spike_t_s, N, float(sim_time / second), float(dt / second),
        SPECTRUM_BATCH_SIZE)


## Plots and saved outputs

In [6]:
def save_run():
    metadata = {
        "run_id": RUN_ID, "parameters": {"N_E": N_E, "N_I": N_I, "C_E": C_E, "C_I": C_I,
        "tau_ms": float(tau/ms), "threshold_mV": float(theta/mV), "reset_mV": float(V_r/mV),
        "refractory_ms": float(tau_rp/ms), "J_mV": float(J/mV), "g": g,
        "delay_ms": float(D/ms), "RI_mV": float(RI/mV), "dt_ms": float(dt/ms),
        "duration_s": float(sim_time/second)}, "seed": SEED,
        "software": {"python": sys.version, "numpy": np.__version__, "scipy": scipy_version, "brian2": brian2_version, "platform": platform.platform()},
        "analysis": {"spectrum": "HOLYKALE dense FFT / 2; batched equivalent", "batch_size": SPECTRUM_BATCH_SIZE},
        "runtime_s": runtime_s, "discarded_transient_s": 0.0}
    np.savez_compressed(RESULTS / f"{RUN_ID}.npz", frequency_hz=frequency_hz, spectrum=spectrum,
                        neuron_rates_hz=neuron_rates, cv_values=cv_values, mean_rate_hz=mean_rate, mean_cv=mean_cv)
    (RESULTS / f"{RUN_ID}.json").write_text(json.dumps(metadata, indent=2))

if RUN_SIMULATION:
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)
    mask = spike_t_s < 0.3
    axes[0].plot(spike_t_s[mask], spike_i[mask], "|", color="deeppink", ms=2)
    axes[0].set(xlabel="Time after start (s)", ylabel="Neuron index", title="Sample raster")
    axes[1].semilogy(frequency_hz, spectrum, color="deeppink")
    axes[1].set(xlabel="Frequency (Hz)", ylabel="Power (FFT units)", title="Mean single-neuron spectrum")
    if SAVE_FIGURES: fig.savefig(FIGURES / f"{RUN_ID}.png", dpi=150)
    plt.show()
    save_run()
